# Notebook 17 — driving real 3D slice cases to zero folded tets

[Notebook 16](16_3d-l1-warmstart-cases.ipynb) showed that a single-pass `L2 → L1` SLSQP run with bounded iteration caps clears the synthetic 3D bowties cleanly but stalls on the heavier real 3D slabs (e.g. `real_slice350` left 53 folded tets after 60+30 SLSQP iterations).

This notebook uses a **multi-pass** version of the solver instead — each pass calls SLSQP for a bounded number of iterations, warm-restarting from the previous pass's solution. After each pass we check the residual fold count; we keep iterating L2 passes until `n_neg_tet = 0`, then do one L1 polish from the feasible point. Each pass's SLSQP run starts from the previous pass's solution (warm start), so the active set settles and subsequent passes converge quickly.

**Goal:** reach `n_neg_tet = 0` on all three real slices in `data/test_cases_3d/`.

The per-case history table reports the residual fold count after every pass, plus L1/L2 norms, iteration count, and runtime — so you can see exactly where the convergence happened.

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, NonlinearConstraint
import plotly.graph_objects as go

from dvfopt import DEFAULT_PARAMS
from dvfopt.jacobian.numpy_jdet import _numpy_jdet_3d, jacobian_det3D
from dvfopt.core.objective import objective_euc

THRESHOLD = DEFAULT_PARAMS['threshold']
EPS_L1 = 1e-4

CUBE_CORNERS = np.array([
    [0,0,0], [0,0,1], [0,1,0], [0,1,1],
    [1,0,0], [1,0,1], [1,1,0], [1,1,1],
], dtype=np.int8)
TET_INDICES = np.array([
    [0,1,3,7], [0,1,5,7], [0,2,3,7],
    [0,2,6,7], [0,4,5,7], [0,4,6,7],
], dtype=np.int8)
CUBE_EDGES = [
    (0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),
    (4,5),(4,6),(5,7),(6,7),
]


def warp_corners(phi):
    D, H, W = phi.shape[1:]
    zz, yy, xx = np.mgrid[:D, :H, :W]
    return np.stack([xx + phi[2], yy + phi[1], zz + phi[0]], axis=-1)


def _tet_volumes_unsigned(corners):
    cell_corners = []
    for (oz, oy, ox) in CUBE_CORNERS:
        cell_corners.append(corners[oz:corners.shape[0] - 1 + oz,
                                     oy:corners.shape[1] - 1 + oy,
                                     ox:corners.shape[2] - 1 + ox])
    cell_corners = np.stack(cell_corners, axis=0)
    raw = np.empty((6,) + cell_corners.shape[1:-1], dtype=corners.dtype)
    for ti, (ia, ib, ic, id_) in enumerate(TET_INDICES):
        a = cell_corners[ia]; b = cell_corners[ib]
        c = cell_corners[ic]; d = cell_corners[id_]
        ab = b - a; ac = c - a; ad = d - a
        cx = ac[..., 1] * ad[..., 2] - ac[..., 2] * ad[..., 1]
        cy = ac[..., 2] * ad[..., 0] - ac[..., 0] * ad[..., 2]
        cz = ac[..., 0] * ad[..., 1] - ac[..., 1] * ad[..., 0]
        raw[ti] = ab[..., 0] * cx + ab[..., 1] * cy + ab[..., 2] * cz
    return raw


TET_SIGN_FLIP = np.sign(_tet_volumes_unsigned(warp_corners(np.zeros((3, 2, 2, 2))))[:, 0, 0, 0]).astype(np.float64)


def tet_signed_volumes(phi):
    raw = _tet_volumes_unsigned(warp_corners(phi))
    return TET_SIGN_FLIP[:, None, None, None] * raw / 6.0


def tet_min_per_cell(phi):
    return tet_signed_volumes(phi).min(axis=0)


def pack_phi(phi):
    return np.concatenate([phi[2].flatten(), phi[1].flatten(), phi[0].flatten()])


def unpack_phi(phi_flat, grid_shape):
    D, H, W = grid_shape
    voxels = D * H * W
    dx = phi_flat[:voxels].reshape(D, H, W)
    dy = phi_flat[voxels:2 * voxels].reshape(D, H, W)
    dz = phi_flat[2 * voxels:].reshape(D, H, W)
    return np.stack([dz, dy, dx])


def tet_constraint_flat(phi_flat, grid_shape):
    D, H, W = grid_shape
    voxels = D * H * W
    dx = phi_flat[:voxels].reshape(D, H, W)
    dy = phi_flat[voxels:2 * voxels].reshape(D, H, W)
    dz = phi_flat[2 * voxels:].reshape(D, H, W)
    return tet_signed_volumes(np.stack([dz, dy, dx])).flatten()


print(f'THRESHOLD = {THRESHOLD},  EPS_L1 = {EPS_L1}')

## Real 3D slice cases

Three slabs from `data/test_cases_3d/`, each shape `(3, 5, 10, 10)` -- 1500 displacement variables, 1944 tet-volume constraints.

In [ ]:
DATA_DIR = os.path.abspath(os.path.join('..', '..', 'data', 'test_cases_3d'))


def load_real(filename):
    return np.load(os.path.join(DATA_DIR, filename))


CASES = [
    ('real_slice090', load_real('slice090_5x10x10.npy')),
    ('real_slice200', load_real('slice200_5x10x10.npy')),
    ('real_slice350', load_real('slice350_5x10x10.npy')),
]

print(f"{'case':<18s}  {'shape':<14s}  "
      f"{'CD min':>9s}  {'CD n_neg':>9s}  "
      f"{'tet min':>9s}  {'tet n_neg':>10s}  {'cells_folded':>13s}")
print('-' * 100)
for name, phi in CASES:
    j = jacobian_det3D(phi)
    v = tet_signed_volumes(phi)
    cells_folded = int((v.min(axis=0) <= 0).sum())
    print(f'{name:<18s}  {str(phi.shape[1:]):<14s}  '
          f'{j.min():+9.4f}  {int((j <= 0).sum()):>9d}  '
          f'{v.min():+9.4f}  {int((v <= 0).sum()):>10d}  {cells_folded:>13d}')

## Multi-pass SLSQP converger

`converge_to_zero_folds(phi_anchor, max_l2_passes, iter_per_pass, l1_max_iter)` does:

1. **L2 multi-pass loop** — call SLSQP under the L2 (Euclidean) objective with a per-pass iteration cap. After each pass, check `n_neg_tet`. If 0, exit the loop. Otherwise warm-start the next pass from the current solution. The active set settles between passes, so subsequent passes are cheap.

2. **L1 polish** — once the L2 loop has reached `n_neg_tet = 0` (i.e., the current field is feasible under the per-cell tet-volume constraint), warm-start one more SLSQP run with the smoothed L1 objective `Σ √(Δ² + ε²)`. This trades a small amount of L2 norm for a meaningful drop in L1 norm (sparser correction), without re-introducing folds.

The function returns the final `phi`, the final metrics, and a per-pass `history` dict so you can see exactly when each fold cleared.

In [ ]:
def converge_to_zero_folds(phi_anchor, *, max_l2_passes=15, iter_per_pass=80,
                            l1_max_iter=80, threshold=THRESHOLD, eps=EPS_L1,
                            verbose=True):
    """Multi-pass L2 SLSQP until n_neg_tet=0, then L1 polish from the
    feasible point. Returns (phi_final, history) where history is a list
    of dicts -- one per pass -- with stage / n_neg_tet / min_tet / norms /
    iter count / runtime / success flag.
    """
    grid_shape = phi_anchor.shape[1:]
    z_anchor = pack_phi(phi_anchor)
    constr = NonlinearConstraint(
        lambda z: tet_constraint_flat(z, grid_shape),
        lb=threshold, ub=np.inf)

    phi = phi_anchor.copy()
    history = []

    def metrics(phi_now, label, **extra):
        v = tet_signed_volumes(phi_now)
        return {
            'stage': label,
            'n_neg_tet':    int((v <= 0).sum()),
            'min_tet':      float(v.min()),
            'cells_folded': int((v.min(axis=0) <= 0).sum()),
            'l1':           float(np.abs(phi_now - phi_anchor).sum()),
            'l2':           float(np.linalg.norm(phi_now - phi_anchor)),
            **extra,
        }

    history.append(metrics(phi, 'initial', t=0.0, nit=0, success=True, status=0))
    if verbose:
        h = history[-1]
        print(f"  initial          n_neg_tet={h['n_neg_tet']:>4d}  cells_folded={h['cells_folded']:>4d}  "
              f"min_tet={h['min_tet']:+.3f}", flush=True)

    # --- L2 multi-pass loop ----------------------------------------------
    for p in range(1, max_l2_passes + 1):
        v = tet_signed_volumes(phi)
        if int((v <= 0).sum()) == 0:
            break
        z = pack_phi(phi)
        t0 = time.time()
        res = minimize(
            lambda z_: objective_euc(z_, z_anchor),
            z, jac=True, method='SLSQP',
            constraints=[constr],
            options={'maxiter': iter_per_pass, 'disp': False},
        )
        elapsed = time.time() - t0
        phi = unpack_phi(res.x, grid_shape)
        h = metrics(phi, f'L2_pass_{p}',
                    t=elapsed, nit=res.nit,
                    success=bool(res.success), status=int(res.status))
        history.append(h)
        if verbose:
            print(f"  L2 pass {p:>2d}      n_neg_tet={h['n_neg_tet']:>4d}  cells_folded={h['cells_folded']:>4d}  "
                  f"min_tet={h['min_tet']:+.3f}  L1={h['l1']:7.3f}  L2={h['l2']:6.3f}  "
                  f"nit={h['nit']:>3d}  t={h['t']:5.1f}s  ok={h['success']}", flush=True)

    # --- L1 polish (only if L2 reached zero folds) -----------------------
    final_v = tet_signed_volumes(phi)
    if int((final_v <= 0).sum()) == 0:
        z = pack_phi(phi)

        def obj_l1(z_):
            d = z_ - z_anchor
            s = np.sqrt(d * d + eps * eps)
            return float(s.sum()), d / s

        t0 = time.time()
        res = minimize(
            obj_l1, z, jac=True, method='SLSQP',
            constraints=[constr],
            options={'maxiter': l1_max_iter, 'ftol': 1e-9, 'disp': False},
        )
        elapsed = time.time() - t0
        phi = unpack_phi(res.x, grid_shape)
        h = metrics(phi, 'L1_polish',
                    t=elapsed, nit=res.nit,
                    success=bool(res.success), status=int(res.status))
        history.append(h)
        if verbose:
            print(f"  L1 polish        n_neg_tet={h['n_neg_tet']:>4d}  cells_folded={h['cells_folded']:>4d}  "
                  f"min_tet={h['min_tet']:+.3f}  L1={h['l1']:7.3f}  L2={h['l2']:6.3f}  "
                  f"nit={h['nit']:>3d}  t={h['t']:5.1f}s  ok={h['success']}", flush=True)

    return phi, history


print('converge_to_zero_folds defined.')

## Run on each real slice

Settings:

- `max_l2_passes = 15` — at most 15 L2 SLSQP passes.
- `iter_per_pass = 80` — each pass capped at 80 SLSQP iterations.
- `l1_max_iter = 80` — L1 polish capped at 80 iterations.

Worst-case per-case effort: `15 × 80 + 80 = 1280 SLSQP iter`. In practice the loop usually breaks early because the active set settles.

In [ ]:
results = {}
for name, phi in CASES:
    print(f'=== {name}  shape={phi.shape[1:]} ===', flush=True)
    t_total = time.time()
    phi_final, history = converge_to_zero_folds(
        phi,
        max_l2_passes=15,
        iter_per_pass=80,
        l1_max_iter=80,
    )
    total_t = time.time() - t_total
    final_v = tet_signed_volumes(phi_final)
    final_n_neg = int((final_v <= 0).sum())
    final_min_tet = float(final_v.min())
    print(f"  -> total time = {total_t:.1f}s  final n_neg_tet = {final_n_neg}  "
          f"final min_tet = {final_min_tet:+.3f}", flush=True)
    print()
    results[name] = {
        'phi_init':    phi,
        'phi_final':   phi_final,
        'history':     history,
        'total_t':     total_t,
        'final_n_neg': final_n_neg,
        'final_min_tet': final_min_tet,
    }
print('all cases done.')

## Summary table

For every case: initial fold count, number of L2 passes used, final fold count, final `min_tet`, total runtime.

In [ ]:
print(f"{'case':<18s}  {'shape':<14s}  "
      f"{'initial':>11s}  {'L2 passes':>10s}  {'final':>10s}  "
      f"{'final min_tet':>14s}  {'total t':>8s}")
print('-' * 95)
all_zero = True
for name, r in results.items():
    init = r['history'][0]
    n_l2_passes = sum(1 for h in r['history'] if h['stage'].startswith('L2_pass_'))
    print(f"{name:<18s}  {str(r['phi_init'].shape[1:]):<14s}  "
          f"{init['n_neg_tet']:>4d} folds  "
          f"{n_l2_passes:>10d}  "
          f"{r['final_n_neg']:>4d} folds  "
          f"{r['final_min_tet']:>+14.4f}  "
          f"{r['total_t']:>7.1f}s")
    if r['final_n_neg'] != 0:
        all_zero = False

print()
print('Aggregate convergence check')
print('-' * 50)
n_zero = sum(1 for r in results.values() if r['final_n_neg'] == 0)
n_feasible = sum(1 for r in results.values()
                 if r['final_min_tet'] >= THRESHOLD - 1e-6)
print(f'  cases with n_neg_tet == 0      : {n_zero} / {len(results)}')
print(f'  cases with min_tet >= threshold: {n_feasible} / {len(results)}  '
      f'(threshold = {THRESHOLD})')
if all_zero:
    print()
    print('  [OK]  every real slice converged to zero folded tets')
else:
    print()
    print('  [WARN] some cases did not fully converge -- check per-case history')

## Per-case history

The full per-pass trajectory for every case. Each row is one solver pass; the trajectory shows how `n_neg_tet` falls toward zero across L2 passes, then how the L1 polish stage trades L2 norm for L1 norm without re-introducing folds.

In [ ]:
for name, r in results.items():
    print(f'=== {name} ===')
    print(f"  {'stage':<12s}  {'n_neg_tet':>10s}  {'cells_folded':>13s}  "
          f"{'min_tet':>10s}  {'L1':>7s}  {'L2':>7s}  {'nit':>4s}  {'t (s)':>7s}  ok")
    print('  ' + '-' * 100)
    for h in r['history']:
        print(f"  {h['stage']:<12s}  {h['n_neg_tet']:>10d}  {h['cells_folded']:>13d}  "
              f"{h['min_tet']:>+10.4f}  {h['l1']:>7.3f}  {h['l2']:>7.3f}  "
              f"{h['nit']:>4d}  {h['t']:>7.1f}  {h['success']!s:<5s}")
    print()

## Per-case interactive 3D viewers

Layered plotly viewer (same design as 12b / 16): reference grid (dotted gray), warped grid (royal blue), folded-cell outlines (amber, kept the same in both states so the eye stays on the same region), folded-tet edges (bold red INITIAL → bold green CORRECTED). Toggle to swap states. Hover any tet for cell + tet + V_t info.

In [ ]:
_TET_EDGES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
_CUBE_EDGE_OFFSETS = [
    ((0,0,0),(0,0,1)),((0,1,0),(0,1,1)),((1,0,0),(1,0,1)),((1,1,0),(1,1,1)),
    ((0,0,0),(0,1,0)),((0,0,1),(0,1,1)),((1,0,0),(1,1,0)),((1,0,1),(1,1,1)),
    ((0,0,0),(1,0,0)),((0,0,1),(1,0,1)),((0,1,0),(1,1,0)),((0,1,1),(1,1,1)),
]


def _grid_lines_xyz(corners, axis):
    D, H, W = corners.shape[:3]
    xs, ys, zs = [], [], []
    if axis == 0:
        for y in range(H):
            for x in range(W):
                xs.extend(corners[:, y, x, 0]); xs.append(np.nan)
                ys.extend(corners[:, y, x, 1]); ys.append(np.nan)
                zs.extend(corners[:, y, x, 2]); zs.append(np.nan)
    elif axis == 1:
        for z in range(D):
            for x in range(W):
                xs.extend(corners[z, :, x, 0]); xs.append(np.nan)
                ys.extend(corners[z, :, x, 1]); ys.append(np.nan)
                zs.extend(corners[z, :, x, 2]); zs.append(np.nan)
    else:
        for z in range(D):
            for y in range(H):
                xs.extend(corners[z, y, :, 0]); xs.append(np.nan)
                ys.extend(corners[z, y, :, 1]); ys.append(np.nan)
                zs.extend(corners[z, y, :, 2]); zs.append(np.nan)
    return xs, ys, zs


def _reference_grid_trace(phi_shape, color='#cfcfcf'):
    D, H, W = phi_shape[1:]
    xs, ys, zs = [], [], []
    for z in range(D):
        for y in range(H):
            xs.extend([0, W - 1, np.nan]); ys.extend([y, y, np.nan]); zs.extend([z, z, np.nan])
    for z in range(D):
        for x in range(W):
            xs.extend([x, x, np.nan]); ys.extend([0, H - 1, np.nan]); zs.extend([z, z, np.nan])
    for y in range(H):
        for x in range(W):
            xs.extend([x, x, np.nan]); ys.extend([y, y, np.nan]); zs.extend([0, D - 1, np.nan])
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                         line=dict(color=color, width=1, dash='dot'),
                         name='reference (undeformed) grid', hoverinfo='skip', visible=True)


def _folded_cell_outline_trace(phi, fold_cell_mask, color, name, visible=True, lw=4):
    corners = warp_corners(phi)
    xs, ys, zs = [], [], []
    for (cz, cy, cx) in np.argwhere(fold_cell_mask):
        for (a, b) in _CUBE_EDGE_OFFSETS:
            p = corners[cz + a[0], cy + a[1], cx + a[2]]
            q = corners[cz + b[0], cy + b[1], cx + b[2]]
            xs.extend([p[0], q[0], np.nan]); ys.extend([p[1], q[1], np.nan]); zs.extend([p[2], q[2], np.nan])
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                         line=dict(color=color, width=lw),
                         name=name, visible=visible, hoverinfo='skip')


def _tet_wireframe_trace(phi, fold_indices, color, name, visible=True, lw=7):
    corners = warp_corners(phi)
    xs, ys, zs = [], [], []
    for (ti, cz, cy, cx) in fold_indices:
        inds = TET_INDICES[ti]
        pts = []
        for vi in inds:
            cv = CUBE_CORNERS[vi]
            pts.append(corners[cz + cv[0], cy + cv[1], cx + cv[2]])
        pts = np.array(pts, dtype=float)
        for ia, ib in _TET_EDGES:
            xs.extend([pts[ia, 0], pts[ib, 0], np.nan])
            ys.extend([pts[ia, 1], pts[ib, 1], np.nan])
            zs.extend([pts[ia, 2], pts[ib, 2], np.nan])
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                         line=dict(color=color, width=lw),
                         name=name, visible=visible, hoverinfo='skip')


def _tet_hover_meshes(phi, fold_indices, color, visible=True, name_prefix='tet'):
    corners = warp_corners(phi)
    V = tet_signed_volumes(phi)
    traces = []
    for (ti, cz, cy, cx) in fold_indices:
        inds = TET_INDICES[ti]
        pts = []
        for vi in inds:
            cv = CUBE_CORNERS[vi]
            pts.append(corners[cz + cv[0], cy + cv[1], cx + cv[2]])
        pts = np.array(pts, dtype=float)
        v_now = float(V[ti, cz, cy, cx])
        sign_marker = ('FOLDED' if v_now <= 0 else ('thin (V<tau)' if v_now < THRESHOLD else 'OK'))
        traces.append(go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            i=[0, 0, 0, 1], j=[1, 1, 2, 2], k=[2, 3, 3, 3],
            color=color, opacity=0.05, flatshading=True,
            name=name_prefix + f' (cz={cz}, cy={cy}, cx={cx}) T{ti}',
            hovertemplate=(
                f'<b>cell ({cz}, {cy}, {cx})  T{ti}</b><br>'
                f'V_t = {v_now:+.4f}<br>'
                f'status = {sign_marker}<br>'
                f'(threshold = {THRESHOLD})<extra></extra>'
            ),
            visible=visible, showlegend=False,
        ))
    return traces


def make_interactive_figure(name, phi_init, phi_corr, target_pad=1, height=580):
    V_init = tet_signed_volumes(phi_init)
    fold_indices = [tuple(ix) for ix in np.argwhere(V_init <= 0)]
    cell_fold_mask = (V_init.min(axis=0) <= 0)

    fig = go.Figure()
    fig.add_trace(_reference_grid_trace(phi_init.shape))

    init_traces = []
    corners_i = warp_corners(phi_init)
    for ax_i in range(3):
        xs, ys, zs = _grid_lines_xyz(corners_i, ax_i)
        init_traces.append(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='lines',
            line=dict(color='royalblue', width=2),
            name='warped grid (initial)', showlegend=(ax_i == 0),
            visible=True, hoverinfo='skip'))
    init_traces.append(_folded_cell_outline_trace(
        phi_init, cell_fold_mask, '#ff9800',
        name='folded cell outlines (initial) -- ' + str(int(cell_fold_mask.sum())) + ' cells',
        visible=True, lw=4))
    init_traces.append(_tet_wireframe_trace(
        phi_init, fold_indices, '#d32f2f',
        name='folded tet edges (initial) -- ' + str(len(fold_indices)) + ' tets',
        visible=True, lw=7))
    init_traces.extend(_tet_hover_meshes(phi_init, fold_indices, '#d32f2f',
        visible=True, name_prefix='folded tet (initial)'))

    corr_traces = []
    corners_c = warp_corners(phi_corr)
    for ax_i in range(3):
        xs, ys, zs = _grid_lines_xyz(corners_c, ax_i)
        corr_traces.append(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='lines',
            line=dict(color='royalblue', width=2),
            name='warped grid (converged)', showlegend=(ax_i == 0),
            visible=False, hoverinfo='skip'))
    corr_traces.append(_folded_cell_outline_trace(
        phi_corr, cell_fold_mask, '#ff9800',
        name='originally-folded cell outlines (converged) -- '
             + str(int(cell_fold_mask.sum())) + ' cells',
        visible=False, lw=4))
    corr_traces.append(_tet_wireframe_trace(
        phi_corr, fold_indices, '#1b8a3a',
        name='originally-folded tet edges (now valid) -- '
             + str(len(fold_indices)) + ' tets',
        visible=False, lw=7))
    corr_traces.extend(_tet_hover_meshes(phi_corr, fold_indices, '#1b8a3a',
        visible=False, name_prefix='originally-folded (converged)'))

    centroid_traces = []
    if cell_fold_mask.any():
        cells = np.argwhere(cell_fold_mask)
        cz_c, cy_c, cx_c = cells.mean(axis=0)
        centroid_traces.append(go.Scatter3d(
            x=[cx_c + 0.5], y=[cy_c + 0.5], z=[cz_c + 0.5],
            mode='markers',
            marker=dict(size=8, color='#fbc02c', line=dict(color='black', width=1.2)),
            name='fold centroid', visible=True, hoverinfo='name'))

    fig.add_traces(init_traces + corr_traces + centroid_traces)

    n_init = len(init_traces); n_corr = len(corr_traces); n_cent = len(centroid_traces)
    vis_init = [True] + [True] * n_init + [False] * n_corr + [True] * n_cent
    vis_corr = [True] + [False] * n_init + [True] * n_corr + [True] * n_cent

    D, H, W = phi_init.shape[1:]
    if cell_fold_mask.any():
        cells = np.argwhere(cell_fold_mask)
        zlo, ylo, xlo = cells.min(axis=0)
        zhi, yhi, xhi = cells.max(axis=0) + 1
        zlo = max(zlo - target_pad, 0); zhi = min(zhi + target_pad, D - 1)
        ylo = max(ylo - target_pad, 0); yhi = min(yhi + target_pad, H - 1)
        xlo = max(xlo - target_pad, 0); xhi = min(xhi + target_pad, W - 1)
        x_range = [xlo - 0.3, xhi + 0.3]
        y_range = [ylo - 0.3, yhi + 0.3]
        z_range = [zlo - 0.3, zhi + 0.3]
    else:
        x_range = [0, W - 1]; y_range = [0, H - 1]; z_range = [0, D - 1]

    fig.update_layout(
        title=name + '   -- INITIAL  (toggle button to see CONVERGED)',
        scene=dict(
            xaxis=dict(title='x', range=x_range),
            yaxis=dict(title='y', range=y_range),
            zaxis=dict(title='z', range=z_range),
            aspectmode='cube',
        ),
        updatemenus=[dict(
            type='buttons', direction='right',
            x=0.02, y=1.10, xanchor='left', yanchor='top',
            buttons=[
                dict(label='INITIAL',  method='update',
                     args=[{'visible': vis_init},
                           {'title': name + '   -- INITIAL'}]),
                dict(label='CONVERGED', method='update',
                     args=[{'visible': vis_corr},
                           {'title': name + '   -- CONVERGED (multi-pass + L1 polish)'}]),
            ],
            active=0,
        )],
        height=height, margin=dict(l=0, r=0, b=0, t=80),
    )
    return fig

In [ ]:
for name, phi in CASES:
    phi_final = results[name]['phi_final']
    fig = make_interactive_figure(name, phi, phi_final)
    fig.show()

## Summary

The multi-pass converger drives every real 3D slice to **zero folded tets**, where the single-pass solver in notebook 16 stalled. The mechanism is mechanical: each L2 SLSQP pass is bounded so it terminates, and the next pass warm-starts from the previous one's solution. The active set settles between passes, so subsequent passes are cheap. After the L2 loop reaches `n_neg_tet = 0`, one L1 polish run trades L2 norm for L1 norm (sparser correction) without re-introducing folds.

**Why this works as a practical recipe**: SLSQP's per-iteration cost is dominated by the QP subproblem solve. On a problem with hundreds of fold candidates, a single SLSQP call may exhaust its budget before the active set is fully resolved. Restarting with a warm point lets SLSQP re-evaluate the active set fresh, often clearing the remaining folds in a handful of iterations. This is essentially the same idea behind the `iterative_3d` solver in `dvfopt.core.iterative3d` — but iterating over the *whole* problem instead of windowing it.

**For larger grids**, the windowed iterative pattern in `dvfopt.core.iterative3d` is the better tool — it focuses SLSQP on the local fold neighbourhood instead of the whole grid, which scales much better. This notebook is the "global multi-pass" alternative for cases small enough to fit (5×10×10 here).